In [1]:
# from utils import *
# from param_search_var_osc import *

import numpy as np
from IPython.display import clear_output
import gzip
import pickle

In [2]:
EJ_subdivisions = 40
EC_subdivisions = 80
EL_subdivisions = 80
Er_subdivisions = 80

EJ_values = np.linspace(4, 16, EJ_subdivisions)
Er_values = np.linspace(3, 18, Er_subdivisions)
g_values = np.array([0.1, 0.3, 0.5, 0.7])
EC_EL_extents = []

total_jobs = 10000
existing_chunk_num = 0
num_chunks_per_EJ = total_jobs // len(EJ_values) # = 250

num_done = 0
for EJ in EJ_values:

    # EC_values = np.linspace(0.5, EJ*3/2, EC_subdivisions)
    EC_values = np.linspace(0.5, EJ/2, EC_subdivisions)

    EL_values = np.linspace(0.05, EJ*0.05+0.2, EL_subdivisions)
    EC_EL_extents.append((EC_values[0], EC_values[-1], EL_values[0], EL_values[-1]))

    ## You don't have to uncomment the following lines
#     EC_grid, EL_grid, Er_grid, g_grid = np.meshgrid(EC_values, EL_values, Er_values,g_values)
#     EC_flat = EC_grid.flatten()
#     EL_flat = EL_grid.flatten()
#     Er_flat = Er_grid.flatten()
#     g_flat = g_grid.flatten()
#     EC_chunks = np.array_split(EC_flat, num_chunks_per_EJ)
#     EL_chunks = np.array_split(EL_flat, num_chunks_per_EJ)
#     Er_chunks = np.array_split(Er_flat, num_chunks_per_EJ)
#     g_chunks = np.array_split(g_flat, num_chunks_per_EJ)
#     for EC_chunk, EL_chunk,Er_chunk,g_chunk in zip(EC_chunks, EL_chunks, Er_chunks,g_chunks):
#         job = search_job(EJ, EC_chunk, EL_chunk,Er_chunk,g_chunk)
#         with open(f'{existing_chunk_num}.pkl', 'wb') as f:
#             pickle.dump(job, f)
#         existing_chunk_num += 1
#     num_done+=1
#     clear_output()
#     print(f"num done:{num_done}/{EJ_subdivisions}")


# def pack_pkl_files_to_zip(zip_filename="param_search.zip"):
#     # Create a new ZIP file
#     print('zipping')
#     with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         # Loop through all files in the current directory
#         for filename in os.listdir('.'):
#             # Check if the file is a .pkl file with an integer name
#             name, ext = os.path.splitext(filename)
#             if ext == '.pkl' and name.isdigit():
#                 # Add the file to the ZIP
#                 zipf.write(filename)
#                 # Delete the .pkl file
#                 os.remove(filename)
#     print('zipping done')
                
# pack_pkl_files_to_zip()


In [3]:


num_elements_in_result = 11 #number of values produced for one set of parameters.
shape_of_grid = (EJ_subdivisions, EC_subdivisions, EL_subdivisions, Er_subdivisions,len(g_values))
list_of_grid = [np.zeros(shape_of_grid,dtype=np.float32) for _ in range(num_elements_in_result)]
# Initialize counter for existing chunks
existing_chunk_num = 0


total_elements = EC_subdivisions * EL_subdivisions* Er_subdivisions* len(g_values)
chunk_len = int(total_elements/ num_chunks_per_EJ)


for EJ_idx, EJ in enumerate(EJ_values):    
    # Initialize flattened arrays to store the results for this EJ value
    list_of_flat = [np.zeros(total_elements) for _ in range(num_elements_in_result)]
    
    flat_idx = 0
    # Read that many chunks for this EJ value
    for _ in range(total_jobs // len(EJ_values)):
        try:
            with gzip.GzipFile(f"param_search_result_osc_v3/result_{existing_chunk_num}.zip", "rb") as f:
                job = pickle.load(f)
                                    
            # Place the results back into the flattened arrays
            for idx, element in enumerate(job.results):
                list_of_flat[idx][flat_idx:flat_idx + chunk_len] = element
            
            # Update the index for the flattened arrays
            flat_idx += chunk_len
        except Exception as e:
            clear_output()
            print(f"Error occurred while loading chunk {existing_chunk_num}: {e}")
            # Place the results back into the flattened arrays
            for idx, element in enumerate(job.results):
                list_of_flat[idx][flat_idx:flat_idx + chunk_len] = np.full(chunk_len, None)

            # Update the index for the flattened arrays
            flat_idx += chunk_len 
        existing_chunk_num += 1
    
    # Reshape the flattened arrays back into the original grid for this EJ value
    for idx, grid in enumerate(list_of_grid):
        grid[EJ_idx, :, :, :, :] = list_of_flat[idx].reshape(shape_of_grid[1:])


In [4]:
# unique_EJ_dict = {}
# unique_EC = set()
# unique_EL = set()
# unique_Er = set()
# unique_g = set()

# for i in range(10000):
#     try:
#         with gzip.GzipFile(f"param_search_result_osc_v3/result_{i}.zip", "rb") as f:
#             job = pickle.load(f)
#         if job.EJ not in unique_EJ_dict:
#             unique_EJ_dict[job.EJ] = [set(), set(), set(), set()]
        
#         unique_EJ_dict[job.EJ][0].update(job.EC_values)
#         unique_EJ_dict[job.EJ][1].update(job.EL_values)
#         unique_EJ_dict[job.EJ][2].update(job.Er_values)
#         unique_EJ_dict[job.EJ][3].update(job.g_values)

#         clear_output()
#         print(i)
#     except:
#         pass

In [23]:
from matplotlib.colors import LogNorm, SymLogNorm
from ipywidgets import interact, FloatSlider
import matplotlib.pyplot as plt
import numpy as np

diff_lamb_norm = LogNorm(vmax = 1e-1,vmin = 1e-4)
detuning_norm = LogNorm(vmax = 1e-1,vmin = 1e-4)
def plot_all(EJ, Er,g):
    EJ_idx = np.argmin(np.abs(EJ_values - EJ))
    Er_idx = np.argmin(np.abs(Er_values - Er))
    g_idx = np.argmin(np.abs(g_values - g))
    EJ = EJ_values[EJ_idx]
    Er = Er_values[Er_idx]
    g = g_values[g_idx]
    plt.figure(figsize=(15, 12))
    
    plt.subplot(4, 5, 1)
    plt.imshow(list_of_grid[0][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=LogNorm(vmax = 1,vmin = 1e-2))
    plt.colorbar(label='')
    plt.title(f'one-two transition for \n EJ = {EJ},\n Er = {Er}')
    
    plt.subplot(4, 5, 2)
    plt.imshow(list_of_grid[-2][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=detuning_norm)
    plt.colorbar(label='')
    plt.title(f'detunning: abs(00->01 - 10->11)')

    plt.subplot(4, 5, 3)
    plt.imshow(list_of_grid[-1][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=detuning_norm)
    plt.colorbar(label='')
    plt.title(f'detunning: abs(00->01 - 20->21)')
    
    plt.subplot(4, 5, 4)
    plt.imshow(list_of_grid[3][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 01->02')
    plt.subplot(4, 5, 5)
    plt.imshow(list_of_grid[4][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 02->03')
    plt.subplot(4, 5, 6)
    plt.imshow(list_of_grid[5][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 03->04')
    plt.subplot(4, 5, 7)
    plt.imshow(list_of_grid[6][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 04->05')
    plt.subplot(4, 5, 8)
    plt.imshow(list_of_grid[7][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 05->06')
    plt.subplot(4, 5, 9)
    plt.imshow(list_of_grid[7][EJ_idx, :, :, Er_idx,g_idx], extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=diff_lamb_norm)
    plt.colorbar(label='')
    plt.title(f'00->01 - 06->07')

    plt.subplot(4, 5, 10)
    plt.imshow(abs(list_of_grid[1][EJ_idx, :, :, Er_idx,g_idx]), extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=LogNorm(vmax = 1e-6,vmin = 2e-1), cmap='coolwarm')
    plt.colorbar(label='')
    plt.title(f'10->20 - 11->21')

    plt.subplot(4, 5, 11)
    plt.imshow(abs(list_of_grid[2][EJ_idx, :, :, Er_idx,g_idx]), extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=LogNorm(vmax = 1e-6,vmin = 2e-1), cmap='coolwarm')
    plt.colorbar(label='')
    plt.title(f'10->20 - 12->22')

    plt.subplot(4, 5, 12)
    plt.imshow(abs(list_of_grid[-2][EJ_idx, :, :, Er_idx,g_idx]-list_of_grid[-1][EJ_idx, :, :, Er_idx,g_idx]), extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=LogNorm(vmax =  1e-6,vmin = 2e-1), cmap='coolwarm')
    plt.colorbar(label='')
    plt.title(f'difference in difference of detunning')

    plt.subplot(4, 5, 13)
    plt.imshow(np.minimum(list_of_grid[-2][EJ_idx, :, :, Er_idx,g_idx], list_of_grid[-1][EJ_idx, :, :, Er_idx,g_idx]), extent=EC_EL_extents[EJ_idx], origin='lower', aspect='auto', norm=LogNorm(vmax =  1e-6,vmin = 2e-1), cmap='coolwarm')
    plt.colorbar(label='')
    plt.xlabel('EC')
    plt.ylabel('EL')
    plt.title(f'min of the two detunnings')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.3, hspace=0.2)
    plt.show()

interact(plot_all, 
         EJ=FloatSlider(min=EJ_values[0], max=EJ_values[-1], step=(EJ_values[1]-EJ_values[0]), value=EJ_values[0]),
         Er=FloatSlider(min=Er_values[0], max=Er_values[-1], step=(Er_values[1]-Er_values[0]), value=Er_values[0]),
         g=FloatSlider(min=g_values[0], max=g_values[-1], step=(g_values[1]-g_values[0]), value=g_values[0]))


interactive(children=(FloatSlider(value=4.0, description='EJ', max=16.0, min=4.0, step=0.3076923076923075), Fl…

<function __main__.plot_all(EJ, Er, g)>

# End of the notebook